In [ ]:
import gc
import re
from pathlib import Path

import matplotlib.pyplot as plt
import nltk
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.stem.snowball import SnowballStemmer
from nltk.tokenize import word_tokenize, sent_tokenize
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader

# По условию используем именно нужную видеокарту.
device = torch.device("cuda:0")
print('device:', device)

PROJECT_DIR = Path('.') if Path('data').exists() else Path('Текущий контроль 7-8')
DATA_DIR = PROJECT_DIR / 'data'

MODELS_DIR = PROJECT_DIR / 'saved_models'
MODELS_DIR.mkdir(exist_ok=True)

criterion = nn.CrossEntropyLoss()


In [ ]:
nltk.download('punkt')
try:
    nltk.download('punkt_tab')
except Exception:
    pass


## 1. Представление и предобработка текстовых данных 

1.1 Операции по предобработке:
* токенизация
* стемминг / лемматизация
* удаление стоп-слов
* удаление пунктуации
* приведение к нижнему регистру
* любые другие операции над текстом

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer('english')


def clear_memory(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def train_classifier(model, train_loader, test_loader, criterion, optimizer, epochs, device):
    model = model.to(device)
    history = {'train_loss': [], 'test_loss': [], 'test_acc': []}

    for epoch in range(epochs):
        model.train()
        train_loss_sum = 0.0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * X_batch.size(0)

        train_loss = train_loss_sum / len(train_loader.dataset)
        test_loss, test_acc = evaluate_classifier(model, test_loader, criterion, device)
        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        print(f'epoch {epoch + 1}/{epochs}: train_loss={train_loss:.4f} test_loss={test_loss:.4f} test_acc={test_acc:.4f}')

    return history


def evaluate_classifier(model, loader, criterion, device):
    model.eval()
    losses = []
    y_true = []
    y_pred = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            losses.append(loss.item() * X_batch.size(0))
            preds = logits.argmax(dim=1)
            y_true.extend(y_batch.cpu().tolist())
            y_pred.extend(preds.cpu().tolist())

    mean_loss = sum(losses) / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)
    return mean_loss, acc


def plot_history(history, title):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_loss'], label='train')
    axes[0].plot(history['test_loss'], label='test')
    axes[0].set_title(title + ' loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(history['test_acc'], label='test_acc', color='green')
    axes[1].set_title(title + ' accuracy')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
text = 'Select your preferences and run the install command. Stable represents the most currently tested and supported version of PyTorch. Note that LibTorch is only available for C++'


def preprocess_text(text: str):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z.!?]+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    processed_tokens = []

    for token in tokens:
        if re.fullmatch(r'[.!?]', token):
            continue
        processed_tokens.append(stemmer.stem(token))

    return processed_tokens

print('processed_tokens:', preprocess_text(text))


Реализовать функцию `preprocess_text(text: str)`, которая:
* приводит строку к нижнему регистру
* заменяет все символы, кроме a-z, A-Z и знаков .,!? на пробел


1.2 Представление текстовых данных при помощи бинарного кодирования


Представить первое предложение из `text` в виде тензора `sentence_t`: `sentence_t[i] == 1`, если __слово__ с индексом `i` присуствует в предложении.

## 2. Классификация фамилий по национальности

Датасет: https://disk.yandex.ru/d/owHew8hzPc7X9Q?w=1

2.1 Считать файл `surnames/surnames.csv`. 

2.2 Закодировать национальности числами, начиная с 0.

2.3 Разбить датасет на обучающую и тестовую выборку

2.4 Реализовать класс `Vocab` (токен = __символ__)

2.5 Реализовать класс `SurnamesDataset`

2.6. Обучить классификатор.

2.7 Измерить точность на тестовой выборке. Проверить работоспособность модели: прогнать несколько фамилий студентов группы через модели и проверить результат. Для каждой фамилии выводить 3 наиболее вероятных предсказания.

In [ ]:
class Vocab:
    def __init__(self, data):
        unique_tokens = sorted(set(data))
        self.idx_to_token = unique_tokens
        self.token_to_idx = {token: idx for idx, token in enumerate(unique_tokens)}
        self.vocab_len = len(unique_tokens)


first_sentence = sent_tokenize(text)[0]
first_sentence_tokens = preprocess_text(first_sentence)
all_tokens = sorted(set(preprocess_text(text)))
word_vocab_demo = Vocab(all_tokens)

sentence_t = torch.zeros(word_vocab_demo.vocab_len, dtype=torch.float32)
for token in first_sentence_tokens:
    sentence_t[word_vocab_demo.token_to_idx[token]] = 1.0

print('first_sentence_tokens:', first_sentence_tokens)
print('sentence_t shape:', sentence_t.shape)
print('sentence_t:', sentence_t)


In [ ]:
class SurnamesDataset(Dataset):
    def __init__(self, X, y, vocab: Vocab):
        self.X = list(X)
        self.y = list(y)
        self.vocab = vocab

    def vectorize(self, surname):
        """Генерирует представление фамилии surname в при помощи бинарного кодирования (см. 1.2)"""
        vector = torch.zeros(self.vocab.vocab_len, dtype=torch.float32)
        for char in surname.lower():
            if char in self.vocab.token_to_idx:
                vector[self.vocab.token_to_idx[char]] = 1.0
        return vector

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.vectorize(self.X[idx]), torch.tensor(self.y[idx], dtype=torch.long)


surnames_path = DATA_DIR / 'surnames.csv'
surnames_df = pd.read_csv(surnames_path)
print('surnames_df columns:', surnames_df.columns.tolist())

label_encoder = LabelEncoder()
surnames_df['target'] = label_encoder.fit_transform(surnames_df['nationality'])

X_train, X_test, y_train, y_test = train_test_split(
    surnames_df['surname'],
    surnames_df['target'],
    test_size=0.2,
    random_state=42,
    stratify=surnames_df['target'],
)

char_vocab = Vocab(''.join(X_train.astype(str).str.lower().tolist()))
train_dataset = SurnamesDataset(X_train.tolist(), y_train.tolist(), char_vocab)
test_dataset = SurnamesDataset(X_test.tolist(), y_test.tolist(), char_vocab)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

class SurnameClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.net(x)

surname_model = SurnameClassifier(char_vocab.vocab_len, 128, len(label_encoder.classes_))
optimizer = optim.Adam(surname_model.parameters(), lr=1e-3)
history_surnames = train_classifier(surname_model, train_loader, test_loader, criterion, optimizer, epochs=20, device=device)
plot_history(history_surnames, 'Surname classifier')

test_loss, test_acc = evaluate_classifier(surname_model, test_loader, criterion, device)
print('Surname classifier accuracy:', test_acc)


def predict_surname(model, surname, vocab, label_encoder, top_k=3):
    model.eval()
    with torch.no_grad():
        vector = train_dataset.vectorize(surname).unsqueeze(0).to(device)
        probs = torch.softmax(model(vector), dim=1).squeeze(0)
        top_probs, top_idx = torch.topk(probs, k=top_k)
        results = []
        for prob, idx in zip(top_probs.cpu().tolist(), top_idx.cpu().tolist()):
            results.append((label_encoder.inverse_transform([idx])[0], prob))
        return results

sample_surnames = ['Ivanov', 'Petrov', 'Smith', 'Kim', 'Garcia']
for surname in sample_surnames:
    print(surname, '->', predict_surname(surname_model, surname, char_vocab, label_encoder))

torch.save(surname_model.state_dict(), MODELS_DIR / 'surname_classifier.pth')
clear_memory(history_surnames, optimizer)


## 3. Классификация обзоров ресторанов

Датасет: https://disk.yandex.ru/d/nY1o70JtAuYa8g

3.1 Считать файл `yelp/raw_train.csv`. Оставить от исходного датасета 10% строчек.

3.2 Воспользоваться функцией `preprocess_text` из 1.1 для обработки текста отзыва. Закодировать рейтинг числами, начиная с 0.

3.3 Разбить датасет на обучающую и тестовую выборку

3.4 Реализовать класс `Vocab` (токен = слово)

3.5 Реализовать класс `ReviewDataset`

3.6 Обучить классификатор

3.7 Измерить точность на тестовой выборке. Проверить работоспособность модели: придумать небольшой отзыв, прогнать его через модель и вывести номер предсказанного класса (сделать это для явно позитивного и явно негативного отзыва)


In [ ]:
class Vocab:
    def __init__(self, data):
        unique_tokens = sorted(set(data))
        self.idx_to_token = unique_tokens
        self.token_to_idx = {token: idx for idx, token in enumerate(unique_tokens)}
        self.vocab_len = len(unique_tokens)


In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, X, y, vocab: Vocab):
        self.X = list(X)
        self.y = list(y)
        self.vocab = vocab

    def vectorize(self, review):
        """Генерирует представление отзыва review при помощи бинарного кодирования (см. 1.2)"""
        vector = torch.zeros(self.vocab.vocab_len, dtype=torch.float32)
        for word in preprocess_text(review):
            if word in self.vocab.token_to_idx:
                vector[self.vocab.token_to_idx[word]] = 1.0
        return vector

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.vectorize(self.X[idx]), torch.tensor(self.y[idx], dtype=torch.long)


yelp_path = DATA_DIR / 'raw_train.csv'
yelp_df = pd.read_csv(yelp_path, header=None, names=['rating', 'text'])
yelp_df = yelp_df.sample(frac=0.1, random_state=42).reset_index(drop=True)
print('yelp_df shape after sampling:', yelp_df.shape)

label_encoder_reviews = LabelEncoder()
yelp_df['target'] = label_encoder_reviews.fit_transform(yelp_df['rating'])

X_train, X_test, y_train, y_test = train_test_split(
    yelp_df['text'],
    yelp_df['target'],
    test_size=0.2,
    random_state=42,
    stratify=yelp_df['target'],
)

train_tokens = []
for review in X_train:
    train_tokens.extend(preprocess_text(str(review)))

word_vocab = Vocab(train_tokens)
review_train_dataset = ReviewDataset(X_train.tolist(), y_train.tolist(), word_vocab)
review_test_dataset = ReviewDataset(X_test.tolist(), y_test.tolist(), word_vocab)

review_train_loader = DataLoader(review_train_dataset, batch_size=64, shuffle=True)
review_test_loader = DataLoader(review_test_dataset, batch_size=128, shuffle=False)

class ReviewClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, output_dim),
        )

    def forward(self, x):
        return self.net(x)

review_model = ReviewClassifier(word_vocab.vocab_len, 256, len(label_encoder_reviews.classes_))
optimizer = optim.Adam(review_model.parameters(), lr=1e-3)
history_reviews = train_classifier(review_model, review_train_loader, review_test_loader, criterion, optimizer, epochs=15, device=device)
plot_history(history_reviews, 'Review classifier')

test_loss, test_acc = evaluate_classifier(review_model, review_test_loader, criterion, device)
print('Review classifier accuracy:', test_acc)


def predict_review(model, review_text, vocab, label_encoder):
    model.eval()
    with torch.no_grad():
        vector = review_train_dataset.vectorize(review_text).unsqueeze(0).to(device)
        pred = model(vector).argmax(dim=1).item()
        return label_encoder.inverse_transform([pred])[0]

positive_review = 'Amazing food, polite staff and fast service. I will definitely come back again.'
negative_review = 'Terrible food, rude staff and a very bad experience. I do not recommend this place.'

print('positive_review ->', predict_review(review_model, positive_review, word_vocab, label_encoder_reviews))
print('negative_review ->', predict_review(review_model, negative_review, word_vocab, label_encoder_reviews))

torch.save(review_model.state_dict(), MODELS_DIR / 'review_classifier.pth')
clear_memory(history_reviews, optimizer)
